# Tokenization

How raw text becomes the integer token IDs a transformer model actually consumes, and what an *attention mask* is for.

**Network note:** this notebook was authored and executed inside a network-restricted sandbox that blocks `huggingface.co` (the same egress policy that blocks Docker Hub in this environment) — so the cell that downloads a real pretrained tokenizer cannot actually run here. It is left in, wrapped so the failure is visible and explicit, and will work unmodified on any machine with normal internet access (a laptop, GitHub Actions, GCP, Colab). Everything else in this notebook is real, executed code with real output.

## 1. What tokenization actually does

A tokenizer turns a string into a sequence of integer IDs from a fixed vocabulary. Modern LLMs use **subword** tokenization (BPE or WordPiece): common whole words get one token, rare words get split into pieces, so the vocabulary stays small (30k-100k entries) while still covering arbitrary text (no 'unknown word' problem).

In [1]:
# A minimal from-scratch tokenizer, built and run entirely offline,
# to make the token-ID / vocabulary / attention-mask mechanics concrete
# before looking at a real subword tokenizer's output.

class ToyWhitespaceTokenizer:
    def __init__(self):
        self.vocab = {'[PAD]': 0, '[UNK]': 1}

    def fit(self, texts):
        for text in texts:
            for word in text.lower().split():
                if word not in self.vocab:
                    self.vocab[word] = len(self.vocab)

    def encode(self, text):
        return [self.vocab.get(w, self.vocab['[UNK]']) for w in text.lower().split()]

corpus = [
    'payment service is failing in production',
    'kubernetes deployment has ImagePullBackOff',
]
tok = ToyWhitespaceTokenizer()
tok.fit(corpus)
print('Vocabulary size:', len(tok.vocab))
print(tok.vocab)

Vocabulary size: 12
{'[PAD]': 0, '[UNK]': 1, 'payment': 2, 'service': 3, 'is': 4, 'failing': 5, 'in': 6, 'production': 7, 'kubernetes': 8, 'deployment': 9, 'has': 10, 'imagepullbackoff': 11}


In [2]:
ids = tok.encode('payment service is failing')
print('token ids:', ids)

# Padding + attention mask: batches need equal-length sequences, so
# shorter sequences are padded, and the attention mask tells the model
# which positions are real tokens (1) vs. padding to ignore (0).
def pad_and_mask(id_lists, pad_id=0):
    max_len = max(len(ids) for ids in id_lists)
    padded, masks = [], []
    for ids in id_lists:
        pad_amount = max_len - len(ids)
        padded.append(ids + [pad_id] * pad_amount)
        masks.append([1] * len(ids) + [0] * pad_amount)
    return padded, masks

batch = [tok.encode(t) for t in corpus]
padded, masks = pad_and_mask(batch)
for text, ids, mask in zip(corpus, padded, masks):
    print(f'{text!r:55} ids={ids} mask={mask}')

token ids: [2, 3, 4, 5]
'payment service is failing in production'              ids=[2, 3, 4, 5, 6, 7] mask=[1, 1, 1, 1, 1, 1]
'kubernetes deployment has ImagePullBackOff'            ids=[8, 9, 10, 11, 0, 0] mask=[1, 1, 1, 1, 0, 0]


## 2. A real subword tokenizer (Hugging Face `transformers`)

This is the real code you'd run in any environment with normal internet access. `AutoTokenizer.from_pretrained(...)` downloads the model's vocabulary/merges files from the Hugging Face Hub the first time (cached locally after that — no repeated downloads, no cost, the model itself is free/open-source).

In [3]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    encoded = tokenizer('Payment service is failing in production', return_tensors=None)
    print('input_ids:', encoded['input_ids'])
    print('attention_mask:', encoded['attention_mask'])
    print('decoded tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids']))
except Exception as exc:
    print('Could not reach huggingface.co from this sandbox (expected here):')
    print(f'  {type(exc).__name__}: {exc}')
    print()
    print('On a normal-network machine this cell prints something like:')
    print("  input_ids: [101, 7909, 2326, 2003, 7989, 1999, 2537, 102]")
    print("  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]")
    print("  decoded tokens: ['[CLS]', 'payment', 'service', 'is', 'failing', 'in', 'production', '[SEP]']")

/home/user/ai-agent/agent-service/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 8a292226-2ec9-4f28-a8bc-39cc8668fa98)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 1s [Retry 1/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 213aa342-0f51-4aa8-90f0-c724f13879b0)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 2s [Retry 2/5].


...

Retrying in 8s [Retry 4/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: e5c43712-d8f2-427a-a7c8-435ea98c9ede)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 8s [Retry 5/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 30008d5c-cb59-4e13-891a-2a21fecc336e)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Could not reach huggingface.co from this sandbox (expected here):
  ProxyError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 30008d5c-cb59-4e13-891a-2a21fecc336e)')

On a normal-network machine this cell prints something like:
  input_ids: [101, 7909, 2326, 2003, 7989, 1999, 2537, 102]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]
  decoded tokens: ['[CLS]', 'payment', 'service', 'is', 'failing', 'in', 'production', '[SEP]']


Notice the real tokenizer adds `[CLS]`/`[SEP]` special tokens the model was trained to expect at sequence boundaries — our toy tokenizer above didn't, because it wasn't trained the way a real model's tokenizer/model pair is.